This notebook is for modeling and evaluating span identification from the SemEval dataset. This is the first filter for propaganda.

In [1]:
import os
import json
import torch
from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, EvalPrediction, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback, RobertaForTokenClassification, RobertaConfig)
from datasets import Dataset
from accelerate.state import AcceleratorState
import zipfile
import shutil
import random
import gdown
import torch.nn.functional as F
from transformers.utils.notebook import NotebookProgressCallback

In [2]:
AcceleratorState._reset_state()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
#Set up paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "semeval_roberta_scanner"
SPECIALIST_DIR = BASE_DIR / "models" / "semeval_roberta_scanner_specialist"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [5]:
google_drive_zip_ID = '1lLqG45VR24QxShlwAfkKB1vG9kHpUsx4'
google_drive_spec_zip_ID = '1O7UmT3L3qfILqdQavAqC7qxzeak86aGX'

In [6]:
#Download models if not already
def setup_models(file_id, target_path):
    target_path = Path(target_path).resolve()
    zip_temp = target_path.with_suffix(".zip")

    #Check if files already exist in the correct spot
    if (target_path / "model.safetensors").exists() or (target_path / "pytorch_model.bin").exists():
        print(f"Model weights detected locally at {target_path}")
        return True

    print(f"Model not found. Preparing {target_path}...")

    #Ensure the specific sub-folder exists
    target_path.mkdir(exist_ok=True, parents=True)

    url = f'https://drive.google.com/uc?id={file_id}'

    try:
        #1. Download the zip
        gdown.download(url, str(zip_temp), quiet=False)

        #2. Extract to a temporary location
        temp_extract = target_path / "temp_extraction"
        if temp_extract.exists(): shutil.rmtree(temp_extract)
        temp_extract.mkdir(parents=True)

        print("Unzipping and cleaning up structure...")
        with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
            members = [m for m in zip_ref.namelist() if "__MACOSX" not in m]
            zip_ref.extractall(temp_extract, members=members)

        #3. Move files from temp_extract into target_path
        for root, dirs, files in os.walk(temp_extract):
            for file in files:
                src_file = Path(root) / file
                dest_file = target_path / file
                shutil.move(str(src_file), str(dest_file))

        #4. Final Cleanup
        shutil.rmtree(temp_extract)
        if zip_temp.exists():
            os.remove(zip_temp)

        print(f"Model files are now in: {target_path}")
        return True

    except Exception as e:
        print(f"Error during setup: {e}")
        if zip_temp.exists(): os.remove(zip_temp)
        return False

In [7]:
#Identify if models exist
base_model_exists = setup_models(google_drive_zip_ID, MODEL_DIR)

Model weights detected locally at /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner


In [8]:
specialist_model_exists = setup_models(google_drive_spec_zip_ID, SPECIALIST_DIR)

Model weights detected locally at /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist


In [9]:
def get_si_metrics(predicted_spans, gold_spans):
    """Official SemEval 2020 Task 11 SI Fuzzy Overlap Math"""
    if not predicted_spans and not gold_spans: return 1.0, 1.0, 1.0
    if not predicted_spans or not gold_spans: return 0.0, 0.0, 0.0

    #Precision calculation
    p_num = 0
    for s in predicted_spans:
        max_overlap = 0
        for t in gold_spans:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            max_overlap = max(max_overlap, intersect / (s[1] - s[0]))
        p_num += max_overlap
    precision = p_num / len(predicted_spans)

    #Recall calculation
    r_num = 0
    for t in gold_spans:
        max_overlap = 0
        for s in predicted_spans:
            intersect = max(0, min(s[1], t[1]) - max(s[0], t[0]))
            max_overlap = max(max_overlap, intersect / (t[1] - t[0]))
        r_num += max_overlap
    recall = r_num / len(gold_spans)

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

In [10]:
#Load article-level span identification data
df_si = pd.read_csv(DATA_DIR / "semeval_si_cleaned.csv")
df_si['propaganda_offsets'] = df_si['propaganda_offsets'].apply(json.loads)
print(f"Loaded {len(df_si)} articles.")
df_si.head()

Loaded 357 articles.


,article_id,text,propaganda_offsets
0,111111111,Next plague outbreak in Madagascar could be 's...,"[[265, 323], [1795, 1935], [149, 157], [1069, ..."
1,111111112,US bloggers banned from entering UK\n\nTwo pro...,"[[191, 219], [476, 556], [785, 798], [958, 101..."
2,111111113,Kate Steinle's death at the hands of a Mexican...,"[[1396, 1430], [3082, 3099], [3828, 3985], [36..."
3,111111114,U.S. judge frees Indonesian immigrant held by ...,"[[1705, 1824]]"
4,111111115,Here are all the sexual misconduct accusations...,"[[658, 700], [1870, 1893], [1655, 1745], [2389..."


In [11]:
#Initialize the model tokenizer
##Tried "bert-base-uncased", Best F1 score after 3 epochs: 0.26392
##"microsoft/deberta-v3-small", After 3 epoches, still getting gradient explosion every time, even after adjusting hyperparameters
raw_dataset = Dataset.from_pandas(df_si)
tokenizer = AutoTokenizer.from_pretrained("roberta-base", add_prefix_space=True)

In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_inputs["overflow_to_sample_mapping"]
    offset_mapping = tokenized_inputs["offset_mapping"]
    labels = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        article_spans = examples["propaganda_offsets"][sample_idx]
        doc_labels = []
        for start, end in offsets:
            if start == end == 0:
                doc_labels.append(-100)
                continue
            is_prop = any(s <= start < e or s < end <= e for s, e in article_spans)
            doc_labels.append(1 if is_prop else 0)
        labels.append(doc_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

#Tokenize and split data
tokenized_datasets = raw_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=raw_dataset.column_names).train_test_split(test_size=0.2, seed=42)

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

In [13]:
#Tried without weighting before and was quickly overfitting, so weight now
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        #Prioritize Recall: Propaganda classes (1, 2) weighted 3x more than background (0)
        weights = torch.tensor([1.0, 3.0, 3.0], device = model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


In [14]:
def compute_metrics(p):
    logits, labels = p
    predictions = np.argmax(logits, axis=2) #Default 0.5 threshold

    #Get correct offsets for current batch
    if len(predictions) == len(tokenized_datasets["test"]):
        eval_offsets = tokenized_datasets["test"]["offset_mapping"]
    else:
        eval_offsets = tokenized_datasets["train"]["offset_mapping"]

    all_p, all_r, all_f1 = [], [], []

    for i in range(len(predictions)):
        pred_spans, gold_spans = [], []
        curr_p, curr_g = None, None

        for j, (pred, label) in enumerate(zip(predictions[i], labels[i])):
            if label == -100: continue
            start, end = eval_offsets[i][j]

            if pred == 1: #Predicted Propaganda
                if curr_p is None: curr_p = [start, end]
                else: curr_p[1] = end
            elif curr_p:
                pred_spans.append(tuple(curr_p)); curr_p = None

            if label == 1: #Gold Propaganda
                if curr_g is None: curr_g = [start, end]
                else: curr_g[1] = end
            elif curr_g:
                gold_spans.append(tuple(curr_g)); curr_g = None

        #Calculate scores for this sentence
        p_val, r_val, f1_val = get_si_metrics(pred_spans, gold_spans)
        all_p.append(p_val)
        all_r.append(r_val)
        all_f1.append(f1_val)

    return {
        "si_precision": np.mean(all_p),
        "si_recall": np.mean(all_r),
        "si_f1": np.mean(all_f1)
    }

In [15]:
#Initialize model - Load from local if exists, else from checkpoint
if (MODEL_DIR / "config.json").exists():
    print(f"Loading existing trained model from: {MODEL_DIR}")
    model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)
    model_already_trained = True
else:
    print(f"No existing model found. Initializing from: {"roberta-base"}")
    model = AutoModelForTokenClassification.from_pretrained("roberta-base", num_labels=3)
    model_already_trained = False

model.to(device)


Loading existing trained model from: ../models/semeval_roberta_scanner


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [16]:
#Set up training arguments with optimized hyperparameters
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.5e-05,
    per_device_train_batch_size=8,
    num_train_epochs=13,
    weight_decay=0.15,
    logging_steps=5,
    metric_for_best_model="si_f1",
    greater_is_better=True,
    dataloader_pin_memory=False,
    disable_tqdm=False,
    report_to="none",
    load_best_model_at_end=True
)

In [17]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [18]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(os.fspath(MODEL_DIR))
    model.config.save_pretrained(MODEL_DIR)
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model loaded from disk. Skipping training.")


config_path = MODEL_DIR / "config.json"
if config_path.exists():
    model = RobertaForTokenClassification.from_pretrained(
        MODEL_DIR,
        local_files_only=True
    ).to(device)
else:
    raise FileNotFoundError(f"Missing config.json in {MODEL_DIR}")

Model loaded from disk. Skipping training.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
#Run evaluation on test set
trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])

print("\n" + "="*30)
print("SI MODEL TEST PERFORMANCE")
print(f"Recall:    {test_results['eval_si_recall']:.4f}")
print(f"Precision: {test_results['eval_si_precision']:.4f}")
print(f"F1 Score:  {test_results['eval_si_f1']:.4f}")
print("="*30)


SI MODEL TEST PERFORMANCE
Recall:    0.6491
Precision: 0.5260
F1 Score:  0.5375


In [20]:
#Get raw model predictions to evaluate how well we did on the various propaganda techniques
test_preds_output = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(test_preds_output.predictions, axis=2)
labels = test_preds_output.label_ids

In [21]:
#Because original mapping dropped columns, we recreate the map to track article IDs
#and split with the exact same seed (42) to guarantee perfectly aligned test rows.
def track_article_ids(examples):
    tokenized = tokenizer(
        examples["text"], truncation=True, max_length=512, stride=128, return_overflowing_tokens=True
    )
    sample_mapping = tokenized["overflow_to_sample_mapping"]
    return {"article_id": [examples["article_id"][i] for i in sample_mapping]}

id_tracker_ds = raw_dataset.map(track_article_ids, batched=True, remove_columns=raw_dataset.column_names)
test_article_ids = id_tracker_ds.train_test_split(test_size=0.2, seed=42)["test"]["article_id"]

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

In [22]:
eval_offsets = tokenized_datasets["test"]["offset_mapping"]

gold_spans_list = []
pred_spans_list = []

#Loop through predictions to extract character offsets
for i in range(len(predictions)):
    art_id = test_article_ids[i]
    curr_p, curr_g = None, None

    for j, (pred, label) in enumerate(zip(predictions[i], labels[i])):
        if label == -100:
            continue
        start, end = eval_offsets[i][j]

        #Track predicted spans
        if pred == 1:
            if curr_p is None: curr_p = [start, end]
            else: curr_p[1] = end
        elif curr_p:
            pred_spans_list.append({"article_id": art_id, "start": curr_p[0], "end": curr_p[1]})
            curr_p = None

        #Track correct spans
        if label == 1:
            if curr_g is None: curr_g = [start, end]
            else: curr_g[1] = end
        elif curr_g:
            gold_spans_list.append({"article_id": art_id, "start": curr_g[0], "end": curr_g[1]})
            curr_g = None

In [23]:
#Drop duplicates caused by sliding window overlaps
gold_df = pd.DataFrame(gold_spans_list).drop_duplicates()
pred_df = pd.DataFrame(pred_spans_list).drop_duplicates()

In [24]:
missed_indices = []
for idx, gold in gold_df.iterrows():
    #A span is missed if no predicted span overlaps with it
    overlaps = pred_df[
        (pred_df['article_id'] == gold['article_id']) &
        (pred_df['start'] < gold['end']) &
        (pred_df['end'] > gold['start'])
    ]
    if len(overlaps) == 0:
        missed_indices.append(idx)

fn_spans = gold_df.loc[missed_indices].copy()
tp_spans = gold_df.drop(missed_indices).copy()

In [25]:
print(f"Total spans in test: {len(gold_df)}")
print(f"Successfully found (TP): {len(tp_spans)}")
print(f"Missed completely (FN): {len(fn_spans)}")

tc_df = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
article_text_map = dict(zip(df_si['article_id'], df_si['text']))

Total spans in test: 1267
Successfully found (TP): 1006
Missed completely (FN): 261


In [26]:
def get_linguistic_features(text):
    return {
        'length': len(text),
        'punct_count': text.count('!') + text.count('?') + text.count('.'),
        'caps_ratio': sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0
    }

def process_spans(df):
    merged_data = []
    ignore_cols = ['article_id', 'text_content', 'span_text', 'start_char', 'end_char', 'sentiment', 'punct_count', 'lexical_diversity']
    technique_cols = [c for c in tc_df.columns if c not in ignore_cols]

    for _, row in df.iterrows():
        art_id = row['article_id']
        g_start = row['start']
        g_end = row['end']

        #Cross-reference with TC dataset using overlaps
        tc_matches = tc_df[
            (tc_df['article_id'] == art_id) &
            (tc_df['start_char'] <= g_end) &
            (tc_df['end_char'] >= g_start)
        ]

        span_text = article_text_map.get(art_id, "")[g_start:g_end]
        feat = get_linguistic_features(span_text)

        if not tc_matches.empty:
            active_techs = tc_matches[technique_cols].max().to_dict()
            merged_data.append({**feat, **active_techs})

    return pd.DataFrame(merged_data)

fn_data = process_spans(fn_spans)
tp_data = process_spans(tp_spans)

In [27]:
print("\n" + "="*60)
print("TOP MOST MISSED TECHNIQUES (Recall by Technique)")
print("="*60)
if not fn_data.empty and not tp_data.empty:
    ignore_cols = ['article_id', 'text_content', 'span_text', 'start_char', 'end_char', 'sentiment', 'punct_count', 'lexical_diversity']
    technique_cols = [c for c in tc_df.columns if c not in ignore_cols]

    #Any stray string columns will become NaN, then 0
    fn_numeric = fn_data[technique_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    tp_numeric = tp_data[technique_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    missed_counts = fn_numeric.sum()
    found_counts = tp_numeric.sum()

    total_counts = found_counts + missed_counts
    valid_techs = total_counts[total_counts > 0].index

    recall_by_tech = (found_counts[valid_techs] / total_counts[valid_techs]).sort_values()

    tech_summary = pd.DataFrame({
        'Recall': recall_by_tech.round(3),
        'Missed Count': missed_counts[valid_techs].astype(int),
        'Found Count': found_counts[valid_techs].astype(int)
    }).sort_values('Recall')

    print(tech_summary.head(10).to_string())

    print("\n" + "="*60)
    print("CHARACTERISTIC COMPARISON: MISSED vs. FOUND")
    print("="*60)
    comparison = pd.concat([
        fn_data[['length', 'punct_count', 'caps_ratio']].mean().rename('Missed Spans (Avg)'),
        tp_data[['length', 'punct_count', 'caps_ratio']].mean().rename('Found Spans (Avg)')
    ], axis=1)
    print(comparison.round(3))
else:
    print("Could not map spans. Ensure 'semeval_tc_cleaned.csv' is available in DATA_DIR.")
print("="*60)


TOP MOST MISSED TECHNIQUES (Recall by Technique)
                                    Recall  Missed Count  Found Count
Thought-terminating_Cliches          0.588             7           10
Repetition                           0.595            60           88
Appeal_to_Authority                  0.645            11           20
Whataboutism_Straw_Men_Red_Herring   0.704             8           19
Causal_Oversimplification            0.705            18           43
Black-and-White_Fallacy              0.769             3           10
Doubt                                0.811            20           86
Flag-Waving                          0.811            14           60
Bandwagon_Reductio_ad_hitlerum       0.833             3           15
Slogans                              0.833             7           35

CHARACTERISTIC COMPARISON: MISSED vs. FOUND
             Missed Spans (Avg)  Found Spans (Avg)
length                    42.90             45.762
punct_count                0.13  

Let's create a second span identification model (a sort of cascade model) that is better at finding some of these rarer techniques.

In [28]:
if not specialist_model_exists:
    print("Specialist model not found. Prepping residual dataset...")

    #Get predictions on train set
    train_output = trainer.predict(tokenized_datasets["train"])
    train_preds = np.argmax(train_output.predictions, axis=2)
    train_labels = train_output.label_ids

    #We need RoBERTa's specific CLS and SEP token IDs to properly cap our new chunks
    sample_input_ids = tokenized_datasets["train"][0]["input_ids"]
    sample_mask = tokenized_datasets["train"][0]["attention_mask"]
    cls_id = sample_input_ids[0]
    sep_id = sample_input_ids[sum(sample_mask) - 1]

    missed_chunks = {"input_ids": [], "attention_mask": [], "labels": []}
    clean_chunks = {"input_ids": [], "attention_mask": [], "labels": []}

    def save_chunk(ids, attn, lbls):
        """Helper to cleanly cap the sequence and sort it into the right bucket."""
        valid_content = [l for l in lbls if l != -100]
        if len(valid_content) == 0:
            return

        if ids[0] != cls_id:
            ids.insert(0, cls_id)
            attn.insert(0, 1)
            lbls.insert(0, -100)

        if ids[-1] != sep_id:
            ids.append(sep_id)
            attn.append(1)
            lbls.append(-100)

        has_missed = any(l > 0 for l in valid_content)
        if has_missed:
            missed_chunks["input_ids"].append(ids)
            missed_chunks["attention_mask"].append(attn)
            missed_chunks["labels"].append(lbls)
        else:
            clean_chunks["input_ids"].append(ids)
            clean_chunks["attention_mask"].append(attn)
            clean_chunks["labels"].append(lbls)

    for i in range(len(tokenized_datasets["train"])):
        input_ids = tokenized_datasets["train"][i]["input_ids"]
        attn_mask = tokenized_datasets["train"][i]["attention_mask"]
        labels = train_labels[i]
        preds = train_preds[i]

        cur_ids, cur_attn, cur_lbls = [], [], []
        in_found_span = False

        for j in range(len(input_ids)):
            if attn_mask[j] == 0:
                continue

            l = labels[j]
            p = preds[j]

            if l != -100:
                in_found_span = (l > 0) and (p > 0)

            if in_found_span:
                if len(cur_ids) > 0:
                    save_chunk(cur_ids, cur_attn, cur_lbls)
                    cur_ids, cur_attn, cur_lbls = [], [], []
            else:
                cur_ids.append(input_ids[j])
                cur_attn.append(attn_mask[j])
                cur_lbls.append(l)

        if len(cur_ids) > 0:
            save_chunk(cur_ids, cur_attn, cur_lbls)

    print(f"-> Extracted {len(missed_chunks['input_ids'])} chunks with missed propaganda.")
    print(f"-> Extracted {len(clean_chunks['input_ids'])} chunks with strictly NO propaganda.")

    missed_list = [{"input_ids": i, "attention_mask": a, "labels": l}
                   for i, a, l in zip(missed_chunks["input_ids"], missed_chunks["attention_mask"], missed_chunks["labels"])]
    clean_list = [{"input_ids": i, "attention_mask": a, "labels": l}
                  for i, a, l in zip(clean_chunks["input_ids"], clean_chunks["attention_mask"], clean_chunks["labels"])]

    sample_size = min(len(missed_list) * 2, len(clean_list))
    random.seed(42)
    sampled_clean = random.sample(clean_list, sample_size)

    final_list = missed_list + sampled_clean
    random.shuffle(final_list)

    final_dict = {
        "input_ids": [ex["input_ids"] for ex in final_list],
        "attention_mask": [ex["attention_mask"] for ex in final_list],
        "labels": [ex["labels"] for ex in final_list]
    }

    residual_train_dataset = Dataset.from_dict(final_dict)
    print(f"Created 'residual_train_dataset' with {len(residual_train_dataset)} continuous sequences ready for the specialist model.")

else:
    print("Specialist model found on disk. Skipping residual dataset prep.")

Specialist model found on disk. Skipping residual dataset prep.


In [29]:
#Check if the folder exists and has files in it
if not os.path.exists(str(SPECIALIST_DIR)) or not os.path.exists(os.path.join(str(SPECIALIST_DIR), "config.json")):
    print("Initializing fresh model for specialist training...")
    specialist_model = AutoModelForTokenClassification.from_pretrained("roberta-base", num_labels=2)
    specialist_model.to(device)

    specialist_args = training_args
    specialist_args.output_dir = SPECIALIST_DIR

    specialist_trainer = WeightedTrainer(
        model=specialist_model,
        args=specialist_args,
        train_dataset=residual_train_dataset,
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
        data_collator=DataCollatorForTokenClassification(tokenizer)
    )

    specialist_trainer.train()
    specialist_trainer.save_model(SPECIALIST_DIR)
    tokenizer.save_pretrained(SPECIALIST_DIR)

else:
    print(f"Loading existing specialist model from: {SPECIALIST_DIR}")
    specialist_model = AutoModelForTokenClassification.from_pretrained(SPECIALIST_DIR)
    specialist_model.to(device)

    specialist_trainer = Trainer(
        model=specialist_model,
        data_collator=DataCollatorForTokenClassification(tokenizer)
    )

Loading existing specialist model from: ../models/semeval_roberta_scanner_specialist


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
print("\nGenerating predictions from both models on the Test Set...")
base_test_output = trainer.predict(tokenized_datasets["test"])
spec_test_output = specialist_trainer.predict(tokenized_datasets["test"])

#1. Get the base model's actual class predictions
base_preds = np.argmax(base_test_output.predictions, axis=2)

print("Merging predictions (Soft Cascade Logic)...")
#Start with base model's raw logits
combined_logits = base_test_output.predictions.copy()

#2. Fix Shape Mismatch: Pad specialist logits from 2 classes to 3
batch_size, seq_len, _ = spec_test_output.predictions.shape
dummy_logits = np.full((batch_size, seq_len, 1), -100.0)
padded_spec_logits = np.concatenate([spec_test_output.predictions, dummy_logits], axis=2)

#3. Calculate Specialist Confidence
#We convert logits to 0.0-1.0 probabilities
spec_probs = F.softmax(torch.tensor(padded_spec_logits), dim=-1).numpy()
propaganda_prob = spec_probs[:, :, 1]

#4. Apply the Soft Cascade
#Threshold = 0.7 means the specialist must be at least 70% sure before we listen to it.
#This prevents the "over-flagging" that kills Precision score
confidence_threshold = 0.9
mask = (base_preds == 0) & (propaganda_prob > confidence_threshold)

#Only swap in the specialist's logits where it cleared the bar
combined_logits[mask] = padded_spec_logits[mask]

print("Calculating final cascade metrics...")
cascade_eval = EvalPrediction(
    predictions=combined_logits,
    label_ids=base_test_output.label_ids
)

combined_metrics = compute_metrics(cascade_eval)

print("\n" + "="*45)
print("CASCADE ENSEMBLE PERFORMANCE (Test Set)")
print("="*45)
for key, value in combined_metrics.items():
    print(f"{key}: {value:.4f}")
print("="*45)


Generating predictions from both models on the Test Set...
